## Kindle Review Sentiment Analysis

#### Best Practises

1. Preprocessing and Cleaning
2. Train Test Split
3. BOW, TFIDF, Word2Vec
4. Train ML algorithms

In [11]:
# load dataset
import pandas as pd

df=pd.read_csv("all_kindle_review.csv")
df.head()

,Unnamed: 0.1,Unnamed: 0,asin,helpful,rating,reviewText,reviewTime,reviewerID,reviewerName,summary,unixReviewTime
0,0,11539,B0033UV8HI,"[8, 10]",3,"Jace Rankin may be short, but he's nothing to ...","09 2, 2010",A3HHXRELK8BHQG,Ridley,Entertaining But Average,1283385600
1,1,5957,B002HJV4DE,"[1, 1]",5,Great short read. I didn't want to put it dow...,"10 8, 2013",A2RGNZ0TRF578I,Holly Butler,Terrific menage scenes!,1381190400
2,2,9146,B002ZG96I4,"[0, 0]",3,I'll start by saying this is the first of four...,"04 11, 2014",A3S0H2HV6U1I7F,Merissa,Snapdragon Alley,1397174400
3,3,7038,B002QHWOEU,"[1, 3]",3,Aggie is Angela Lansbury who carries pocketboo...,"07 5, 2014",AC4OQW3GZ919J,Cleargrace,very light murder cozy,1404518400
4,4,1776,B001A06VJ8,"[0, 1]",4,I did not expect this type of book to be in li...,"12 31, 2012",A3C9V987IQHOQD,Rjostler,Book,1356912000


In [12]:
df = df[['reviewText', 'rating']]
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",3
1,Great short read. I didn't want to put it dow...,5
2,I'll start by saying this is the first of four...,3
3,Aggie is Angela Lansbury who carries pocketboo...,3
4,I did not expect this type of book to be in li...,4


In [13]:
df.shape

(12000, 2)

In [14]:
df.isnull().sum()

reviewText    0
rating        0
dtype: int64

In [15]:
df['rating'].unique()

array([3, 5, 4, 2, 1])

In [16]:
df['rating'].value_counts()

rating
5    3000
4    3000
3    2000
2    2000
1    2000
Name: count, dtype: int64

In [17]:
# Preprocessing

In [18]:
# positive review is 1 and negative review is 0
df['rating'] = df['rating'].apply(lambda x: 0 if x < 3 else 1)

In [19]:
df.head()

,reviewText,rating
0,"Jace Rankin may be short, but he's nothing to ...",1
1,Great short read. I didn't want to put it dow...,1
2,I'll start by saying this is the first of four...,1
3,Aggie is Angela Lansbury who carries pocketboo...,1
4,I did not expect this type of book to be in li...,1


In [21]:
df['rating'].value_counts()

rating
1    8000
0    4000
Name: count, dtype: int64

In [22]:
# 1. Lowercase
df['reviewText'] = df['reviewText'].str.lower()

In [23]:
df.head()

,reviewText,rating
0,"jace rankin may be short, but he's nothing to ...",1
1,great short read. i didn't want to put it dow...,1
2,i'll start by saying this is the first of four...,1
3,aggie is angela lansbury who carries pocketboo...,1
4,i did not expect this type of book to be in li...,1


In [25]:
import re
import nltk
from nltk.corpus import stopwords
nltk.download('stopwords')
from bs4 import BeautifulSoup


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Ilya\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [27]:
## Removing special characters
df['reviewText']=df['reviewText'].apply(lambda x:re.sub('[^a-z A-z 0-9-]+', '',x))
## Remove the stopswords
df['reviewText']=df['reviewText'].apply(lambda x:" ".join([y for y in x.split() if y not in stopwords.words('english')]))
## Remove url 
df['reviewText']=df['reviewText'].apply(lambda x: re.sub(r'(http|https|ftp|ssh)://([\w_-]+(?:(?:\.[\w_-]+)+))([\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])?', '' , str(x)))
## Remove html tags
df['reviewText']=df['reviewText'].apply(lambda x: BeautifulSoup(x, 'html.parser').get_text())
## Remove any additional spaces
df['reviewText']=df['reviewText'].apply(lambda x: " ".join(x.split()))

In [28]:
df.head()

,reviewText,rating
0,jace rankin may short hes nothing mess man hau...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four books wasnt expect...,1
3,aggie angela lansbury carries pocketbooks inst...,1
4,expect type book library pleased find price right,1


In [29]:
# Lemmatization
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
lemmatizer = WordNetLemmatizer()

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Ilya\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [30]:
df['reviewText'] = df['reviewText'].apply(lambda x: " ".join([lemmatizer.lemmatize(word) for word in x.split()]))

In [31]:
df.head()

,reviewText,rating
0,jace rankin may short he nothing mess man haul...,1
1,great short read didnt want put read one sitti...,1
2,ill start saying first four book wasnt expecti...,1
3,aggie angela lansbury carry pocketbook instead...,1
4,expect type book library pleased find price right,1


In [32]:
# Train test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['reviewText'], df['rating'], test_size=0.2)

In [36]:
from sklearn.feature_extraction.text import CountVectorizer
bow=CountVectorizer()
X_train_bow=bow.fit_transform(X_train).toarray()
X_test_bow=bow.transform(X_test).toarray()

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf=TfidfVectorizer()
X_train_tfidf=tfidf.fit_transform(X_train).toarray()
X_test_tfidf=tfidf.transform(X_test).toarray()

In [43]:
import gensim
from gensim.models import Word2Vec, KeyedVectors
import gensim.downloader as api

wv = api.load('word2vec-google-news-300')

In [ ]:
from nltk import sent_tokenize
from gensim.utils import simple_preprocess

words = []

for sentence in X_train:
    sentence_token= sent_tokenize(sentence)
    for word in sentence_token:
        words.append(simple_preprocess(word))

words_test = []

for sentence in X_test:
    sentence_token= sent_tokenize(sentence)
    for word in sentence_token:
        words_test.append(simple_preprocess(word))

In [51]:
words

[['cover',
  'blurb',
  'great',
  'premise',
  'awesome',
  'writing',
  'decent',
  'wasnt',
  'satisfied',
  'done',
  'lacked',
  'certain',
  'something',
  'character',
  'depth',
  'little',
  'time',
  'spent',
  'connection',
  'maybe'],
 ['happy',
  'ever',
  'ever',
  'onethis',
  'good',
  'read',
  'short',
  'story',
  'loving',
  'series',
  'worth',
  'money'],
 ['career',
  'journalist',
  'nearly',
  'year',
  'im',
  'tough',
  'impress',
  'come',
  'reporting',
  'knd',
  'favorite',
  'read',
  'resource',
  'mine',
  'since',
  'got',
  'kindle',
  'year',
  'ago',
  'estimate',
  'get',
  'book',
  'value',
  'due',
  'blog',
  'free',
  'book',
  'alert',
  'per',
  'monthwindwalker',
  'know',
  'publishing',
  'ebook',
  'publishing',
  'technology',
  'inside',
  'he',
  'thorough',
  'update',
  'blog',
  'much',
  'time',
  'daily',
  'breaking',
  'news',
  'thing',
  'kindlehis',
  'opinion',
  'indie',
  'publishing',
  'ebook',
  'publishing',
  'certa

In [52]:
model=gensim.models.Word2Vec(words)

In [53]:
def avg_word2vec(doc):
    vectors = [model.wv[word] for word in doc if word in model.wv.index_to_key]
    if len(vectors) > 0:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.wv.vector_size)

In [59]:
from tqdm import tqdm
import numpy as np

X_train_w2v=[]

for doc in tqdm(words):
    X_train_w2v.append(avg_word2vec(doc))

X_test_w2v=[]

for doc in tqdm(words_test):
    X_test_w2v.append(avg_word2vec(doc))


100%|██████████| 9600/9600 [00:14<00:00, 674.10it/s]


NameError: name 'words_test' is not defined

In [57]:
from sklearn.naive_bayes import GaussianNB
nb_model_bow=GaussianNB().fit(X_train_bow,y_train)
nb_model_tfidf=GaussianNB().fit(X_train_tfidf,y_train)
nb_model_w2v=GaussianNB().fit(X_train_w2v,y_train)

In [41]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [58]:
y_pred_bow=nb_model_bow.predict(X_test_bow)
y_pred_tfidf=nb_model_tfidf.predict(X_test_tfidf)
y_pred_w2v=nb_model_w2v.predict(X_test_w2v)
print("Naive Bayes with BoW")
print("Accuracy:", accuracy_score(y_test, y_pred_bow))
print("Classification Report:\n", classification_report(y_test, y_pred_bow))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_bow))
print("Naive Bayes with TF-IDF")
print("Accuracy:", accuracy_score(y_test, y_pred_tfidf))
print("Classification Report:\n", classification_report(y_test, y_pred_tfidf))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_tfidf))
print("Naive Bayes with Word2Vec")
print("Accuracy:", accuracy_score(y_test, y_pred_w2v))
print("Classification Report:\n", classification_report(y_test, y_pred_w2v))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_w2v))

NameError: name 'X_test_w2v' is not defined